# 🏥 HealthBridge_Al — BERT Training
### Multilingual symptom-disease classifier
**GPU Runtime zaroori hai → Runtime > Change Runtime Type > T4 GPU**

In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ❌'}")
assert torch.cuda.is_available(), "GPU nahi mila! Runtime > Change Runtime Type > T4 GPU"

In [ ]:
!pip install transformers datasets accelerate scikit-learn -q
print("✅ Dependencies installed")

In [ ]:
from google.colab import files
import os

os.makedirs("data/layer2", exist_ok=True)
os.makedirs("data", exist_ok=True)

print("Upload karo: combined_training_data.csv")
uploaded = files.upload()
for fname, data in uploaded.items():
    with open(f"data/layer2/{fname}", "wb") as f:
        f.write(data)
    print(f"✅ Saved: data/layer2/{fname}")

print("\nUpload karo: mapping.json")
uploaded2 = files.upload()
for fname, data in uploaded2.items():
    with open(f"data/{fname}", "wb") as f:
        f.write(data)
    print(f"✅ Saved: data/{fname}")

In [ ]:
import pandas as pd
import json

df = pd.read_csv("data/layer2/combined_training_data.csv")
print(f"Total rows    : {len(df)}")
print(f"Columns       : {list(df.columns)}")
print(f"Unique diseases: {df['label'].nunique()}")
print(f"\nSample data:")
print(df.head(3))

with open("data/mapping.json") as f:
    label2id = json.load(f)

id2label = {int(v): k for k,v in label2id.items()}
total_classes = len(label2id)
print(f"\n✅ Labels loaded: {total_classes} diseases")

In [ ]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Clean data
df = df.dropna(subset=["text", "label"])
df["text"] = df["text"].astype(str).str.strip()
df["label"] = df["label"].str.strip().str.title()
df["label_id"] = df["label"].map(label2id)
df = df.dropna(subset=["label_id"])
df["label_id"] = df["label_id"].astype(int)

# Split
train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42
)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")

# Tokenize
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_ds = Dataset.from_pandas(
    train_df[["text","label_id"]].rename(columns={"label_id":"labels"})
)
val_ds = Dataset.from_pandas(
    val_df[["text","label_id"]].rename(columns={"label_id":"labels"})
)

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)

train_ds.set_format("torch", columns=["input_ids","attention_mask","labels"])
val_ds.set_format("torch",   columns=["input_ids","attention_mask","labels"])

print("✅ Dataset tokenized and ready")

In [ ]:
from transformers import (AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=total_classes,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

training_args = TrainingArguments(
    output_dir="models/bert_healthbridge",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

print("🚀 Training shuru ho rahi hai...")
trainer.train()

In [ ]:
results = trainer.evaluate()
print("━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"✅ Final Accuracy : {results['eval_accuracy']:.2%}")
print(f"✅ Final F1 Score : {results['eval_f1']:.2%}")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━")

model.save_pretrained("models/bert_healthbridge/final")
tokenizer.save_pretrained("models/bert_healthbridge/final")
print("✅ Model saved!")

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("bert_healthbridge_final", "zip", "models/bert_healthbridge/final")
print("📦 Zip created: bert_healthbridge_final.zip")
files.download("bert_healthbridge_final.zip")
print("✅ Download shuru ho gaya!")